# LINZ AWS Data Access with Rasterio

This notebook demonstrates how to directly access and process LINZ raster datasets from AWS S3 using rasterio, without downloading files first.

**Key advantages:**
- 🚀 **Direct data access** - No need to download full files
- 🎯 **Spatial subsetting** - Extract only the data you need
- 📊 **Metadata extraction** - Get raster information instantly
- 🔍 **Overview support** - Use different resolution levels
- 💾 **Memory efficient** - Process large datasets without local storage

## Prerequisites

Install required dependencies:
```bash
pip install rasterio boto3
```

## 1. Import Dependencies and Setup

In [ ]:
from __future__ import annotations

import os
from dataclasses import dataclass
from typing import Optional, Tuple

import setup_gdal_env  # Configure GDAL environment
import rasterio
from rasterio.session import AWSSession
from rasterio.windows import from_bounds
import numpy as np

# Optional: For listing S3 objects
try:
    import boto3
    from botocore import UNSIGNED
    from botocore.config import Config

    BOTO3_AVAILABLE = True
except ImportError:
    BOTO3_AVAILABLE = False
    print("⚠️  boto3 not available. Install with: pip install boto3")

print("✅ Dependencies imported successfully!")
print(f"📦 Rasterio version: {rasterio.__version__}")
if BOTO3_AVAILABLE:
    print(f"📦 Boto3 available for S3 listing")

## 2. Define LINZ Dataset Configuration

In [ ]:
@dataclass(frozen=True)
class DatasetInfo:
    bucket: str
    region: str


# Available LINZ datasets
LINZ_DATASETS: dict[str, DatasetInfo] = {
    "imagery": DatasetInfo(bucket="nz-imagery", region="ap-southeast-2"),
    "elevation": DatasetInfo(bucket="nz-elevation", region="ap-southeast-2"),
    "coastal": DatasetInfo(bucket="nz-coastal", region="ap-southeast-2"),
}

print("🗺️  Available LINZ datasets:")
for name, info in LINZ_DATASETS.items():
    print(f"   {name}: {info.bucket} ({info.region})")

## 3. Core Functions for Rasterio AWS Access

In [ ]:
def build_s3_url(bucket: str, path: str) -> str:
    """Build S3 URL for rasterio access."""
    return f"s3://{bucket}/{path}"


def setup_aws_session() -> AWSSession:
    """Create an AWS session for unsigned requests (public data)."""
    return AWSSession(
        requester_pays=False,
        aws_unsigned=True,
    )


def read_raster_info(s3_url: str, session: AWSSession) -> dict:
    """Read basic raster metadata without loading pixel data."""
    with rasterio.Env(session=session):
        with rasterio.open(s3_url) as dataset:
            info = {
                "driver": dataset.driver,
                "width": dataset.width,
                "height": dataset.height,
                "count": dataset.count,
                "dtype": dataset.dtypes[0],
                "crs": dataset.crs,
                "transform": dataset.transform,
                "bounds": dataset.bounds,
                "res": dataset.res,
                "nodata": dataset.nodata,
                "overviews": [
                    dataset.overviews(i) for i in range(1, dataset.count + 1)
                ],
            }
            return info


print("✅ Core rasterio functions defined!")

## 4. Set Up AWS Session and Test Connection

In [ ]:
# Set up AWS session for unsigned access
session = setup_aws_session()
print("🔌 AWS session configured for unsigned access to public data")

# Test file path - Taranaki imagery
dataset_name = "imagery"
dataset = LINZ_DATASETS[dataset_name]
test_file_path = "taranaki/taranaki_2022-2023_0.1m/rgb/2193/BQ31_10000_0101.tif"

s3_url = build_s3_url(dataset.bucket, test_file_path)
print(f"🎯 Test file S3 URL: {s3_url}")

## 5. Read Raster Metadata (Without Downloading)

In [ ]:
# Read metadata from the test file
print(f"📋 Reading metadata from: {test_file_path.split('/')[-1]}")

try:
    info = read_raster_info(s3_url, session)

    print("\n📊 === Raster Information ===")
    print(f"   Driver: {info['driver']}")
    print(f"   Size: {info['width']:,} x {info['height']:,} pixels")
    print(f"   Bands: {info['count']}")
    print(f"   Data type: {info['dtype']}")
    print(f"   CRS: {info['crs']}")
    print(f"   Resolution: {info['res'][0]:.6f} x {info['res'][1]:.6f}")
    print(f"   Bounds: {info['bounds']}")
    print(f"   NoData: {info['nodata']}")

    if any(info["overviews"]):
        print(f"   Overviews: {info['overviews'][0]}")
        print(f"   📈 Overview levels available: {len(info['overviews'][0])}")
    else:
        print("   Overviews: None")

    # Calculate file size estimate
    pixels = info["width"] * info["height"] * info["count"]
    if "uint8" in str(info["dtype"]):
        bytes_per_pixel = 1
    elif "uint16" in str(info["dtype"]) or "float16" in str(info["dtype"]):
        bytes_per_pixel = 2
    else:
        bytes_per_pixel = 4

    estimated_size_mb = (pixels * bytes_per_pixel) / (1024 * 1024)
    print(f"   📏 Estimated uncompressed size: ~{estimated_size_mb:.1f} MB")

except Exception as e:
    print(f"❌ Error reading raster info: {e}")
    print("💡 Try a different file path or check your internet connection.")

## 6. Read Raster Data Function

In [ ]:
def read_raster_window(
    s3_url: str,
    session: AWSSession,
    bbox: Optional[Tuple[float, float, float, float]] = None,
    overview_level: int = 0,
) -> Tuple[np.ndarray, dict]:
    """Read raster data from S3, optionally clipped to a bounding box.

    Args:
        s3_url: S3 URL to the raster file
        session: AWS session
        bbox: Optional bounding box (minx, miny, maxx, maxy)
        overview_level: Overview level to read (0=full resolution)

    Returns:
        Tuple of (numpy array, metadata dict)
    """
    with rasterio.Env(session=session):
        with rasterio.open(s3_url) as dataset:
            if bbox is not None:
                # Read only the window that intersects with the bbox
                window = from_bounds(*bbox, dataset.transform)
                # Clip window to dataset bounds
                window = window.intersection(
                    rasterio.windows.Window(0, 0, dataset.width, dataset.height)
                )
                data = dataset.read(
                    window=window,
                    out_shape=(dataset.count, int(window.height), int(window.width)),
                )
                # Calculate the transform for the windowed data
                transform = rasterio.windows.transform(window, dataset.transform)
            else:
                # Read full dataset at specified overview level
                if overview_level > 0 and any(dataset.overviews(1)):
                    # Get overview at specified level
                    overview_factor = dataset.overviews(1)[
                        min(overview_level - 1, len(dataset.overviews(1)) - 1)
                    ]
                    out_shape = (
                        dataset.count,
                        dataset.height // overview_factor,
                        dataset.width // overview_factor,
                    )
                    data = dataset.read(out_shape=out_shape)
                    # Adjust transform for overview
                    transform = dataset.transform * dataset.transform.scale(
                        overview_factor
                    )
                else:
                    data = dataset.read()
                    transform = dataset.transform

            metadata = {
                "crs": dataset.crs,
                "transform": transform,
                "width": data.shape[2] if len(data.shape) == 3 else data.shape[1],
                "height": data.shape[1] if len(data.shape) == 3 else data.shape[0],
                "count": data.shape[0] if len(data.shape) == 3 else 1,
                "dtype": data.dtype,
                "nodata": dataset.nodata,
            }

            return data, metadata


print("✅ Raster reading function defined!")

## 7. Example: Read a Small Overview

In [ ]:
# Read data at a lower resolution (overview level 1)
print("📖 Reading raster data at overview level 1 (lower resolution)...")

try:
    data, metadata = read_raster_window(
        s3_url=s3_url,
        session=session,
        overview_level=1,  # Use first overview level
    )

    print("\n📊 === Data Information ===")
    print(f"   Data shape: {data.shape}")
    print(f"   Data type: {data.dtype}")
    print(f"   Data range: {data.min():.2f} to {data.max():.2f}")
    print(f"   Mean value: {data.mean():.2f}")
    print(f"   Memory usage: {data.nbytes / (1024 * 1024):.2f} MB")

    print("\n🗺️  === Spatial Metadata ===")
    print(f"   CRS: {metadata['crs']}")
    print(f"   Transform: {metadata['transform']}")
    print(f"   Width: {metadata['width']} pixels")
    print(f"   Height: {metadata['height']} pixels")

    # Basic statistics for each band
    if len(data.shape) == 3:  # Multi-band
        print(f"\n📈 === Band Statistics ===")
        for i in range(data.shape[0]):
            band_data = data[i]
            print(
                f"   Band {i + 1}: min={band_data.min()}, max={band_data.max()}, mean={band_data.mean():.1f}"
            )

except Exception as e:
    print(f"❌ Error reading raster data: {e}")
    print("💡 The file might not exist or there could be a connection issue.")

## 8. Example: Read a Spatial Subset (Bounding Box)

In [ ]:
# Define a small bounding box in the same CRS as the data
# These coordinates should be within the raster bounds
print("✂️  Reading a spatial subset using bounding box...")

try:
    # First, get the full bounds to understand the coordinate system
    full_info = read_raster_info(s3_url, session)
    full_bounds = full_info["bounds"]

    print(f"📍 Full raster bounds: {full_bounds}")

    # Create a smaller bounding box in the center
    center_x = (full_bounds.left + full_bounds.right) / 2
    center_y = (full_bounds.bottom + full_bounds.top) / 2

    # Create a 1000x1000 unit box around the center
    bbox_size = 500  # Half-size in coordinate units
    bbox = (
        center_x - bbox_size,  # minx
        center_y - bbox_size,  # miny
        center_x + bbox_size,  # maxx
        center_y + bbox_size,  # maxy
    )

    print(f"🎯 Reading subset with bbox: {bbox}")

    subset_data, subset_metadata = read_raster_window(
        s3_url=s3_url, session=session, bbox=bbox
    )

    print("\n📊 === Subset Results ===")
    print(f"   Subset shape: {subset_data.shape}")
    print(f"   Original shape would be: {full_info['width']} x {full_info['height']}")
    print(
        f"   Data reduction: {subset_data.size / (full_info['width'] * full_info['height'] * full_info['count']):.1%}"
    )
    print(f"   Memory usage: {subset_data.nbytes / (1024 * 1024):.2f} MB")

    # Calculate actual bounds of subset
    transform = subset_metadata["transform"]
    width = subset_metadata["width"]
    height = subset_metadata["height"]

    subset_bounds = rasterio.transform.array_bounds(height, width, transform)
    print(f"   Actual subset bounds: {subset_bounds}")

except Exception as e:
    print(f"❌ Error reading subset: {e}")
    print("💡 The bounding box might be outside the raster extent.")

## 9. Save Raster Function

In [ ]:
def save_raster(
    data: np.ndarray,
    metadata: dict,
    output_path: str,
) -> None:
    """Save numpy array as a local raster file."""
    with rasterio.open(
        output_path,
        "w",
        driver="GTiff",
        height=metadata["height"],
        width=metadata["width"],
        count=metadata["count"],
        dtype=metadata["dtype"],
        crs=metadata["crs"],
        transform=metadata["transform"],
        nodata=metadata["nodata"],
        compress="lzw",  # Use LZW compression
        tiled=True,  # Create tiled TIFF
    ) as dst:
        if len(data.shape) == 3:
            # Multi-band
            dst.write(data)
        else:
            # Single band
            dst.write(data, 1)


# Example: Save the subset we just read
if "subset_data" in locals():
    output_file = "rasterio_subset_example.tif"
    print(f"💾 Saving subset to: {output_file}")

    save_raster(subset_data, subset_metadata, output_file)

    # Check file size
    if os.path.exists(output_file):
        file_size = os.path.getsize(output_file) / (1024 * 1024)
        print(f"✅ Saved! File size: {file_size:.2f} MB")

else:
    print("ℹ️  No subset data available to save. Run the previous cell first.")

## 10. Advanced Example: Compare Different Overview Levels

In [ ]:
# Compare different overview levels
print("🔍 Comparing different overview levels...")

overview_comparison = []

for level in range(0, 4):  # Try levels 0-3
    try:
        print(f"\n📊 Testing overview level {level}...")
        data, metadata = read_raster_window(
            s3_url=s3_url, session=session, overview_level=level
        )

        size_mb = data.nbytes / (1024 * 1024)

        comparison = {
            "level": level,
            "shape": data.shape,
            "size_mb": size_mb,
            "total_pixels": data.size,
            "resolution_factor": 2**level if level > 0 else 1,
        }
        overview_comparison.append(comparison)

        print(f"   Shape: {data.shape}")
        print(f"   Memory: {size_mb:.2f} MB")
        print(f"   Pixels: {data.size:,}")

    except Exception as e:
        print(f"   ❌ Level {level} not available: {e}")
        break

if overview_comparison:
    print(f"\n📈 === Overview Level Comparison ===")
    print(
        f"{'Level':<6} {'Width':<8} {'Height':<8} {'Memory (MB)':<12} {'Reduction':<10}"
    )
    print("-" * 50)

    base_pixels = overview_comparison[0]["total_pixels"]
    for comp in overview_comparison:
        reduction = f"{comp['total_pixels'] / base_pixels:.1%}"
        width = comp["shape"][2] if len(comp["shape"]) == 3 else comp["shape"][1]
        height = comp["shape"][1] if len(comp["shape"]) == 3 else comp["shape"][0]
        print(
            f"{comp['level']:<6} {width:<8} {height:<8} {comp['size_mb']:<12.2f} {reduction:<10}"
        )

## 11. Listing S3 Objects with Boto3 (Optional)

In [ ]:
def list_s3_objects(
    bucket: str, prefix: str, region: str, limit: int = 20
) -> list[str]:
    """List objects in S3 bucket using boto3."""
    if not BOTO3_AVAILABLE:
        print("❌ boto3 not available. Install with: pip install boto3")
        return []

    try:
        s3 = boto3.client(
            "s3", region_name=region, config=Config(signature_version=UNSIGNED)
        )

        response = s3.list_objects_v2(Bucket=bucket, Prefix=prefix, MaxKeys=limit)
        objects = []

        if "Contents" in response:
            objects.extend([obj["Key"] for obj in response["Contents"]])

        return objects

    except Exception as e:
        print(f"❌ Error listing S3 objects: {e}")
        return []


# Example: List some raster files
if BOTO3_AVAILABLE:
    print("📂 Listing available raster files...")

    dataset = LINZ_DATASETS["imagery"]
    prefix = "taranaki/taranaki_2022-2023_0.1m/rgb/2193/"

    objects = list_s3_objects(dataset.bucket, prefix, dataset.region, limit=10)

    # Filter for raster formats
    raster_objects = [
        obj
        for obj in objects
        if obj.lower().endswith((".tif", ".tiff", ".jp2", ".png", ".jpg", ".jpeg"))
    ]

    if raster_objects:
        print(f"\n🗺️  Found {len(raster_objects)} raster files:")
        for i, obj in enumerate(raster_objects[:5], 1):  # Show first 5
            filename = obj.split("/")[-1]
            print(f"   {i}. {filename}")

        if len(raster_objects) > 5:
            print(f"   ... and {len(raster_objects) - 5} more files")
    else:
        print("❌ No raster files found with the current prefix.")
else:
    print("ℹ️  Install boto3 to enable S3 object listing functionality.")

## Summary

This notebook demonstrated how to:

✅ **Direct raster access** from AWS S3 using rasterio  
✅ **Read metadata** without downloading files  
✅ **Spatial subsetting** using bounding boxes  
✅ **Multi-resolution access** using overview levels  
✅ **Memory-efficient processing** of large datasets  
✅ **Save processed results** to local files  

### Key Advantages of This Approach:

🚀 **Speed**: Only download the data you need  
💾 **Storage**: No need for local copies of large files  
🎯 **Precision**: Extract exact spatial and temporal subsets  
📊 **Scalability**: Handle datasets larger than local storage  
🔍 **Flexibility**: Different resolution levels for different needs  

### Best Practices:

1. **Start with metadata** to understand the dataset
2. **Use overviews** for exploration and low-resolution analysis
3. **Extract subsets** rather than reading full datasets
4. **Leverage COG format** which is optimized for this workflow
5. **Handle exceptions** gracefully for robust applications

### Next Steps:

- Integrate with your geospatial analysis workflow
- Combine with other libraries (matplotlib, folium, etc.) for visualization
- Build automated processing pipelines
- Explore temporal datasets and time series analysis